使用 Fashion-MNIST 构建两层隐藏层的 MLP；实现训练、验证、测试；在 GPU 上训练；绘制 Loss 和 Accuracy 曲线；保存与加载模型。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Dataset
# 图像数据的处理
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

In [2]:
# 定义数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为张量
    transforms.Normalize((0.5,), (0.5,))  # 标准化
])

In [3]:
data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
#dataset = Dataset.DataLoader(data, batch_size=64, shuffle=True)
train_data,val_data = Dataset.random_split(data,[55000,5000], generator=torch.Generator().manual_seed(42))

test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader=Dataset.DataLoader(train_data,batch_size=64,shuffle=True)
val_loader=Dataset.DataLoader(val_data,batch_size=64,shuffle=False)
test_loader=Dataset.DataLoader(test_data,batch_size=64,shuffle=False)

print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))  # 输出数据集大小


55000 5000 10000


In [4]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(784, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 10)
        #self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [5]:
model = SimpleNN()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()  # 多分类交叉熵损失
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
def run_epoch(model,loader,criterion,optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0
    correct = 0
    total = 0
    with torch.set_grad_enabled(training):
        for images,labels in loader:
            images = images.to(device).flatten(start_dim=1)  # 展平为 (batch_size, 784)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size   #nn.CrossEntropyLoss() 默认返回当前批次的平均损失。
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += batch_size

    return total_loss / total, correct / total

In [9]:
epochs = 3
best_val_acc = -1.0
model_path = './MNIST_model/model.pt'

for epoch in range(epochs):
    train_loss,train_acc=run_epoch(model,train_loader,criterion,optimizer)
    val_loss,val_acc=run_epoch(model,val_loader,criterion)

    print(f"Epoch [{epoch+1}/{epochs}], "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    if val_acc>best_val_acc:
        best_val_acc=val_acc
        torch.save(model.state_dict(),model_path)
        print(f"Model saved with val acc: {best_val_acc:.4f}")

Epoch [1/3], Train Loss: 0.3066, Train Acc: 0.9100, Val Loss: 0.3039, Val Acc: 0.9110
Model saved with val acc: 0.9110
Epoch [2/3], Train Loss: 0.2804, Train Acc: 0.9177, Val Loss: 0.3203, Val Acc: 0.9060
Epoch [3/3], Train Loss: 0.2617, Train Acc: 0.9220, Val Loss: 0.2635, Val Acc: 0.9210
Model saved with val acc: 0.9210


In [10]:
test_model = SimpleNN().to(device)
weight_path = './MNIST_model/model.pt'
weights = torch.load(weight_path, map_location=device,weights_only=True)
test_model.load_state_dict(weights)

test_loss,test_acc=run_epoch(test_model,test_loader,criterion)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Test Loss: 0.2492, Test Acc: 0.9245
